# N02 — Word Embeddings & Semantic NLP

## From bag-of-words to meaning

TF-IDF treats "car" and "automobile" as completely different words. Word embeddings encode semantic similarity: similar words are close in vector space.

## The distributional hypothesis

> "You shall know a word by the company it keeps." — J.R. Firth (1957)

Words that appear in similar contexts have similar meanings. This is the foundation of all word embeddings.

## spaCy's word vectors

The `en_core_web_sm` model has 300-dimensional word vectors (GloVe-based) for ~20,000 common words. Each token's `.vector` attribute gives its embedding.

**A document vector** is typically the mean of its token vectors — a simple but effective baseline.

**Reference:** [spaCy vectors](https://spacy.io/usage/linguistic-features#vectors-similarity)


In [ ]:
import spacy
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import warnings
warnings.filterwarnings('ignore')

nlp = spacy.load('en_core_web_sm')

# 4-class newsgroups
categories = ['sci.med', 'sci.space', 'rec.sport.hockey', 'talk.politics.guns']
train_data = fetch_20newsgroups(subset='train', categories=categories,
                                  remove=('headers','footers','quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                  remove=('headers','footers','quotes'))

X_train_raw, y_train = train_data.data, train_data.target
X_test_raw,  y_test  = test_data.data,  test_data.target
target_names = train_data.target_names

print(f"Train: {len(X_train_raw)} | Test: {len(X_test_raw)}")
print(f"Vector size: {nlp('hello').vector.shape[0]}")
print(f"\nWord similarity examples:")
for w1, w2 in [('car','automobile'), ('doctor','hospital'), ('space','orbit'), ('gun','weapon')]:
    t1, t2 = nlp(w1)[0], nlp(w2)[0]
    if t1.has_vector and t2.has_vector:
        sim = t1.similarity(t2)
        print(f"  {w1} ↔ {w2}: {sim:.3f}")

---
## Exercise 1 — Document Vectors: Mean Pooling

## The Math

The simplest document embedding is the **mean of token vectors**:
$$\vec{d} = \frac{1}{|d|} \sum_{t \in d} \vec{w}_t$$

Only tokens with vectors contribute (skip OOV words). This is called **mean pooling**.

**Weighted mean pooling:** Weight each word vector by its TF-IDF score — common words (stopwords) get low weight.
$$\vec{d}_{\text{weighted}} = \frac{\sum_t \text{tfidf}(t,d) \cdot \vec{w}_t}{\sum_t \text{tfidf}(t,d)}$$

**Task:**
1. Implement `mean_document_vectors(texts)` using spaCy's `nlp.pipe`.
2. Implement `tfidf_weighted_document_vectors(texts, fitted_vectorizer)` — weight each word's vector by its TF-IDF score in that document.
3. Train a LogisticRegression on both. Compare to TF-IDF bag-of-words.
4. Return `embedding_comparison` DataFrame.

In [ ]:
def mean_document_vectors(texts: list, batch_size: int = 64) -> np.ndarray:
    """
    Compute mean word vector per document using spaCy.
    Returns (n_docs, 96) array. Docs with no vectors get zero vector.
    """
    # YOUR CODE HERE
    # Use nlp.pipe for efficiency
    # doc.vector returns the mean of token vectors (already computed by spaCy)
    pass

def tfidf_weighted_vectors(texts: list, vectorizer: TfidfVectorizer,
                             batch_size: int = 64) -> np.ndarray:
    """
    TF-IDF weighted mean document vectors.
    Returns (n_docs, vector_dim) array.
    """
    # YOUR CODE HERE
    # 1. Get TF-IDF weights for each document
    # 2. For each doc: weighted sum of vectors / sum of weights
    # 3. Handle OOV words and zero-weight cases
    pass

print("Computing document vectors (this may take ~30 seconds)...")
X_tr_mean = mean_document_vectors(X_train_raw)
X_te_mean = mean_document_vectors(X_test_raw)
print(f"Document vectors shape: {X_tr_mean.shape}")

In [ ]:
# --- ASSERTIONS ---
assert X_tr_mean is not None
assert X_tr_mean.shape == (len(X_train_raw), nlp.vocab.vectors_length)
assert X_te_mean.shape == (len(X_test_raw), nlp.vocab.vectors_length)
assert not np.isnan(X_tr_mean).any(), "No NaN values allowed"

# Train classifier on mean vectors
clf_mean = LogisticRegression(max_iter=1000, random_state=42)
clf_mean.fit(X_tr_mean, y_train)
acc_mean = accuracy_score(y_test, clf_mean.predict(X_te_mean))

# TF-IDF baseline for comparison
tfidf_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
tfidf_pipe.fit(X_train_raw, y_train)
acc_tfidf = accuracy_score(y_test, tfidf_pipe.predict(X_test_raw))

print(f"✓ Exercise 1 passed")
print(f"Mean vectors: {acc_mean:.4f} | TF-IDF: {acc_tfidf:.4f}")

---
## Exercise 2 — Vector Arithmetic & Semantic Operations

## The Math

The famous example: $\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$

This works because embeddings encode semantic relationships as linear directions in vector space. Gender is roughly a consistent direction.

**Cosine similarity:**
$$\cos(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\|\vec{b}\|}$$

**Task:**
1. Implement `word_analogy(word_a, word_b, word_c, top_k=5)` that finds words closest to $\vec{a} - \vec{b} + \vec{c}$.
2. Test on: `doctor - man + woman`, `paris - france + italy`, `hockey - sport + game`.
3. Implement `find_nearest_words(vector, top_k=10)` using cosine similarity over the full vocabulary.
4. Compute the cosine similarity matrix for a set of medical terms. Show clustering.

In [ ]:
def cosine_similarity_vec(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    # YOUR CODE HERE
    pass

def find_nearest_words(vector: np.ndarray, top_k: int = 10,
                        exclude_words: list = None) -> list:
    """
    Find top_k nearest words to a vector using cosine similarity.
    Returns list of (word, similarity) tuples.
    Only considers words that have vectors.
    """
    # YOUR CODE HERE
    # Iterate over nlp.vocab with has_vector=True
    # Compute cosine similarity
    # Return top_k, excluding words in exclude_words
    pass

def word_analogy(word_a: str, word_b: str, word_c: str, top_k: int = 5) -> list:
    """
    word_a is to word_b as word_c is to ?
    Returns nearest words to vec(a) - vec(b) + vec(c).
    """
    # YOUR CODE HERE
    pass

# Test analogies
for a, b, c in [('doctor', 'man', 'woman'), ('paris', 'france', 'germany')]:
    result = word_analogy(a, b, c, top_k=3)
    if result:
        print(f"{a} - {b} + {c} ≈ {[r[0] for r in result]}")

In [ ]:
# --- ASSERTIONS ---
tok_a = nlp('king')[0]; tok_b = nlp('queen')[0]
if tok_a.has_vector and tok_b.has_vector:
    sim = cosine_similarity_vec(tok_a.vector, tok_b.vector)
    assert 0 < sim < 1, "Cosine similarity must be in (0,1)"
    assert sim > 0.5, "King and queen should be similar (> 0.5)"

# find_nearest_words sanity check
tok_space = nlp('space')[0]
if tok_space.has_vector:
    nearest = find_nearest_words(tok_space.vector, top_k=5)
    if nearest:
        assert len(nearest) <= 5
        assert all(isinstance(r, tuple) and len(r) == 2 for r in nearest)
        assert all(0 <= r[1] <= 1 for r in nearest)

print(f"✓ Exercise 2 passed")
print(f"king ↔ queen similarity: {cosine_similarity_vec(nlp('king')[0].vector, nlp('queen')[0].vector):.3f}")

---
## Exercise 3 — Sentence Similarity & Semantic Search

**Task:** Build a semantic search engine using document vectors — finds documents by meaning, not keywords.

1. Build a `SemanticSearchIndex` class that:
   - Indexes documents using mean pooling
   - Supports `search(query, top_k=5)` — encode query and return nearest docs by cosine similarity
   - Uses a **normalized matrix multiplication** for fast batch similarity: `X_norm @ X_doc_norm.T`

2. Compare search quality: keyword search vs semantic search.
   - Query: `'astronaut orbiting planet'`
   - A good semantic search should return sci.space docs even if they don't contain these exact words.

3. Compute precision@5 for both methods on 20 test queries.

In [ ]:
class SemanticSearchIndex:
    """
    Semantic search using document mean vectors and cosine similarity.
    """
    def __init__(self):
        self.doc_vectors_ = None   # (n_docs, dim) — L2 normalized
        self.documents_ = None
        self.labels_ = None

    def index(self, documents: list, labels: np.ndarray = None) -> 'SemanticSearchIndex':
        """
        Build index from documents.
        L2-normalize vectors for fast cosine via matrix multiplication.
        """
        # YOUR CODE HERE
        pass

    def search(self, query: str, top_k: int = 5) -> list:
        """
        Returns (doc_idx, score) list for top_k matches.
        """
        # YOUR CODE HERE
        # 1. Encode query: nlp(query).vector then L2-normalize
        # 2. scores = query_vec @ self.doc_vectors_.T
        # 3. Return top_k indices and scores
        pass

semantic_index = SemanticSearchIndex()
semantic_index.index(X_train_raw[:500], labels=y_train[:500])

In [ ]:
# --- ASSERTIONS ---
assert semantic_index.doc_vectors_ is not None
# Vectors must be L2-normalized
norms = np.linalg.norm(semantic_index.doc_vectors_, axis=1)
valid = norms > 0
assert np.allclose(norms[valid], 1.0, atol=1e-5)

# Search test
results = semantic_index.search('astronaut orbiting planet', top_k=5)
assert results is not None and len(results) == 5
assert all(0 <= r[1] <= 1 for r in results)

# Top result should mostly be sci.space (category 1)
top_cats = [semantic_index.labels_[r[0]] for r in results]
space_idx = target_names.index('sci.space')
space_hits = sum(c == space_idx for c in top_cats)
print(f"✓ Exercise 3 passed — Space docs in top 5 for space query: {space_hits}/5")

---
## Exercise 4 — SVD/LSA: Latent Semantic Analysis

## The Math

**Latent Semantic Analysis (LSA)** applies SVD to the TF-IDF matrix to find latent semantic dimensions:
$$X = U \Sigma V^T$$

The reduced representation $\tilde{X} = U_k \Sigma_k$ captures the $k$ most important semantic axes. Documents that share the same topics end up close in this space — even without shared words.

This is **exactly** PCA on the TF-IDF matrix, but implemented via SVD (numerically better for sparse matrices).

**Task:**
1. Apply `TruncatedSVD` (LSA) to TF-IDF matrix with varying `n_components` (50, 100, 200, 300).
2. Train LogisticRegression on LSA features for each setting.
3. Compare to: raw TF-IDF, mean word vectors.
4. Inspect the top 10 words for each LSA component — what concept does each capture?
5. Return `lsa_comparison` DataFrame.

In [ ]:
# YOUR CODE HERE
lsa_comparison = None

In [ ]:
# --- ASSERTIONS ---
assert lsa_comparison is not None
assert len(lsa_comparison) == 4
acc_col = [c for c in lsa_comparison.columns if 'acc' in c.lower()][0]
comp_col = [c for c in lsa_comparison.columns if 'comp' in c.lower() or 'n_' in c.lower()][0]
assert lsa_comparison[acc_col].max() > 0.70
print(f"✓ Exercise 4 passed")
print(lsa_comparison.to_string(index=False))

---
## Exercise 5 — Named Entity Recognition: Extraction Pipeline

**Task:** Build an entity extraction and analysis pipeline using spaCy.

1. Extract all named entities from the training corpus.
2. Build `entity_frequency_df`: entity_text, entity_label, frequency, in_n_documents — for top 50 entities.
3. Build **entity-based features**: for each document, a vector of entity type counts (PERSON, ORG, GPE, etc.).
4. Train a classifier using ONLY entity features. Does it beat random?
5. Find the entities that are most discriminative between categories (highest chi² score).

In [ ]:
def extract_entities(texts: list, batch_size: int = 64) -> list:
    """
    Extract entities from each document.
    Returns list of lists: [[('entity_text', 'LABEL'), ...], ...]
    """
    # YOUR CODE HERE
    pass

def entity_type_features(entity_lists: list) -> np.ndarray:
    """
    Build per-document feature vector of entity type counts.
    Entity types: PERSON, ORG, GPE, LOC, PRODUCT, EVENT, DATE, MONEY, NORP
    Returns (n_docs, n_entity_types) array.
    """
    # YOUR CODE HERE
    pass

print("Extracting entities (may take ~30 seconds)...")
train_entities = extract_entities(X_train_raw[:300])
print(f"Entities extracted from {len(train_entities)} docs")

In [ ]:
# --- ASSERTIONS ---
assert len(train_entities) == 300
assert all(isinstance(e, list) for e in train_entities)
# Each entity should be (text, label)
for doc_ents in train_entities[:5]:
    for ent in doc_ents:
        assert len(ent) == 2, "Entity must be (text, label) tuple"

X_ent = entity_type_features(train_entities)
if X_ent is not None:
    assert X_ent.shape[0] == 300
    assert X_ent.shape[1] >= 5  # at least 5 entity types
    assert (X_ent >= 0).all()

print(f"✓ Exercise 5 passed — Entity feature shape: {X_ent.shape if X_ent is not None else 'None'}")

---
## Exercise 6 — Text Clustering with Embeddings

**Task:** Cluster documents using their vector representations and evaluate cluster quality.

1. Use the mean document vectors from Exercise 1.
2. Apply K-Means (k=4) in the embedding space.
3. Compute cluster purity: for each cluster, find the majority category. Purity = fraction of documents matching majority.
4. Compare: K-Means on TF-IDF (dense via TruncatedSVD 100d) vs K-Means on word vectors.
5. Implement `cluster_purity(labels_true, labels_pred)` from scratch.
6. Return `clustering_comparison` DataFrame.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def cluster_purity(labels_true: np.ndarray, labels_pred: np.ndarray) -> float:
    """
    Cluster purity: fraction of points assigned to their majority category.
    For each cluster: count majority class. Purity = sum(majority_counts) / n.
    """
    # YOUR CODE HERE
    pass

# YOUR CODE HERE
clustering_comparison = None

In [ ]:
# --- ASSERTIONS ---
# Purity test
perfect_purity = cluster_purity(np.array([0,0,1,1]), np.array([0,0,1,1]))
assert abs(perfect_purity - 1.0) < 1e-10, "Perfect clustering: purity=1"
random_purity = cluster_purity(np.array([0,1,0,1]), np.array([0,0,1,1]))
assert 0 < random_purity < 1

assert clustering_comparison is not None
purity_col = [c for c in clustering_comparison.columns if 'purity' in c.lower()][0]
assert (clustering_comparison[purity_col] > 0.4).all(), "Purity should be > 40%"
print(f"✓ Exercise 6 passed")
print(clustering_comparison.to_string(index=False))

---
## Exercise 7 — Combining Features: TF-IDF + Embeddings

**Task:** Often the best approach combines sparse (TF-IDF) and dense (embedding) features.

1. Build three feature sets:
   - Sparse only: TF-IDF 10,000 features
   - Dense only: mean word vectors (96d)
   - Combined: concatenate both (after scaling dense features to unit variance)
2. Train LinearSVC on all three.
3. Also try: LSA 100d + mean vectors (both dense, concatenated).
4. Report accuracy, macro F1, training time.
5. Return `feature_combination_results` DataFrame.

In [ ]:
from scipy.sparse import hstack, csr_matrix
import time

# YOUR CODE HERE
feature_combination_results = None

In [ ]:
# --- ASSERTIONS ---
assert feature_combination_results is not None
assert len(feature_combination_results) >= 3
acc_col = [c for c in feature_combination_results.columns if 'acc' in c.lower()][0]
assert (feature_combination_results[acc_col] > 0.60).all()
print(f"✓ Exercise 7 passed")
print(feature_combination_results.to_string(index=False))

---
## Exercise 8 — Sentiment Analysis Pipeline

**Task:** Build a sentiment classifier using only spaCy embeddings — no task-specific training data.

**Approach: Lexicon + embeddings hybrid**
1. Create a small sentiment lexicon: 20 positive words, 20 negative words.
2. For each word in a document, find its semantic similarity to the positive/negative prototype vectors (mean of positive/negative word vectors).
3. Document sentiment score = mean(positive similarities) - mean(negative similarities).

Use the IMDB movie review dataset (via sklearn) and evaluate.

4. Compare: lexicon-based embedding score vs TF-IDF + LogReg.
5. Return `sentiment_result` dict.

In [ ]:
# Use a simpler binary classification proxy from 20 newsgroups
# sci.med (0) vs talk.politics.guns (1) — different sentiment/tone
binary_cats = ['sci.med', 'talk.politics.guns']
binary_train = fetch_20newsgroups(subset='train', categories=binary_cats,
                                    remove=('headers','footers','quotes'))
binary_test  = fetch_20newsgroups(subset='test',  categories=binary_cats,
                                    remove=('headers','footers','quotes'))

POSITIVE_WORDS = ['good', 'positive', 'benefit', 'improve', 'health', 'help',
                   'safe', 'effective', 'research', 'science']
NEGATIVE_WORDS = ['dangerous', 'weapon', 'kill', 'violent', 'harm', 'threat',
                   'attack', 'crime', 'gun', 'shoot']

def lexicon_sentiment_score(text: str, pos_words: list,
                              neg_words: list) -> float:
    """
    Compute sentiment score using word vector similarity to lexicon prototypes.
    Returns scalar: positive = similar to positive prototype, negative = negative.
    """
    # YOUR CODE HERE
    # 1. Compute positive prototype = mean vector of pos_words
    # 2. Compute negative prototype = mean vector of neg_words
    # 3. For each token in text: compute similarity to both prototypes
    # 4. Return mean(pos_sims) - mean(neg_sims)
    pass

# YOUR CODE HERE
sentiment_result = None

In [ ]:
# --- ASSERTIONS ---
score = lexicon_sentiment_score('The medicine is effective and safe.', POSITIVE_WORDS, NEGATIVE_WORDS)
if score is not None:
    assert isinstance(score, float)

assert sentiment_result is not None
assert 'lexicon_accuracy' in sentiment_result
assert 'tfidf_accuracy' in sentiment_result
print(f"✓ Exercise 8 passed")
print(f"Lexicon accuracy: {sentiment_result['lexicon_accuracy']:.4f}")
print(f"TF-IDF accuracy: {sentiment_result['tfidf_accuracy']:.4f}")

---
## Exercise 9 — Text Summarization: Extractive

## The Math

**Extractive summarization:** select the most important sentences from the original text. No text is generated — just selected.

**TextRank for sentences:**
1. Represent each sentence as its mean word vector
2. Build similarity graph: edge weight = cosine similarity between sentence vectors
3. Run PageRank on the graph
4. Return top-k sentences by PageRank score (in original order)

**Task:** Implement extractive summarization. Evaluate by comparing compressed ratio (characters in summary / characters in original) vs information retention (classifier trained on original, applied to summary).

In [ ]:
def extractive_summarize(text: str, top_k: int = 3,
                           d: float = 0.85, n_iter: int = 30) -> str:
    """
    TextRank extractive summarization.
    1. Split into sentences using spaCy
    2. Embed each sentence (mean word vectors)
    3. Build similarity matrix
    4. Run PageRank
    5. Return top_k sentences in original order
    """
    # YOUR CODE HERE
    pass

# Test on a long document
long_doc = ' '.join(X_train_raw[:3])  # concatenate 3 docs
summary = extractive_summarize(long_doc, top_k=3)
if summary:
    print(f"Original: {len(long_doc)} chars")
    print(f"Summary: {len(summary)} chars ({len(summary)/len(long_doc):.1%})")
    print(f"\nSummary:\n{summary[:300]}...")

In [ ]:
# --- ASSERTIONS ---
assert summary is not None and isinstance(summary, str)
assert len(summary) < len(long_doc), "Summary must be shorter than original"
assert len(summary) > 0, "Summary must not be empty"
# Summary sentences should be actual sentences from the original
sentences = [s.text.strip() for s in nlp(long_doc).sents if len(s.text.strip()) > 20]
summary_sentences = [s.strip() for s in summary.split('.') if len(s.strip()) > 10]
print(f"✓ Exercise 9 passed — Compression: {len(summary)/len(long_doc):.1%}")

---
## Exercise 10 — Capstone: Full NLP Feature Engineering Pipeline

**Spec:** Build the best NLP classifier combining everything from N01 and N02.

**Feature engineering pipeline:**
1. **Lexical features**: TF-IDF unigrams + bigrams (10k features)
2. **Semantic features**: mean word vectors (96d)
3. **Linguistic features**: POS distribution, sentence length, entity density
4. **Topic features**: LDA topic distribution (20 topics)
5. **Readability features**: avg word length, vocabulary richness (unique/total tokens)

**Task:**
1. Build all 5 feature sets and concatenate.
2. Feature selection: remove near-zero variance features.
3. Train a GradientBoostingClassifier on the full feature set.
4. Compare against TF-IDF-only baseline.
5. Feature importance: which feature group contributes most?
6. Return `nlp_capstone_result` dict.

In [ ]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import LatentDirichletAllocation
import lightgbm as lgb

def compute_readability_features(texts: list) -> np.ndarray:
    """
    Compute readability features per document.
    Returns (n_docs, n_features) array.
    Features: avg_word_len, vocab_richness, avg_sentence_len, doc_length
    """
    # YOUR CODE HERE
    pass

# YOUR CODE HERE: build full pipeline
nlp_capstone_result = None

In [ ]:
# --- ASSERTIONS ---
assert nlp_capstone_result is not None
required = ['full_pipeline_accuracy', 'tfidf_baseline_accuracy',
            'feature_group_importance', 'n_features_total']
for k in required:
    assert k in nlp_capstone_result, f"Missing: {k}"

assert nlp_capstone_result['full_pipeline_accuracy'] > 0.70
assert nlp_capstone_result['n_features_total'] > 100

print(f"✓ Exercise 10 passed — Full NLP pipeline complete")
print(f"Full pipeline: {nlp_capstone_result['full_pipeline_accuracy']:.4f}")
print(f"TF-IDF baseline: {nlp_capstone_result['tfidf_baseline_accuracy']:.4f}")
print(f"Total features: {nlp_capstone_result['n_features_total']}")
print(f"Feature group importance:")
for grp, imp in nlp_capstone_result['feature_group_importance'].items():
    print(f"  {grp}: {imp:.4f}")